In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import scipy.sparse as sp
import warnings
# Plotting options, change to your liking
sc.logging.print_header()
sc.settings.set_figure_params(dpi=300, frameon=False, figsize=(5, 5))
sc.settings.verbosity = 0
warnings.simplefilter(action="ignore", category=FutureWarning)

In [ ]:
# annotated scRNA reference data was obtained from https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE268494
# and was converted to anndata (GSE268494_PKDaggr_GEO_count_RNA.h5ad) object using script: convert_h5ad.R

In [ ]:
# read data and add annotation and umap embedding
adata = sc.read('GSE268494_PKDaggr_GEO_count_RNA.h5ad')
meta = pd.read_csv('GSE268494_PKDaggr_GEO_metadata.csv', index_col=0)
adata.obs = meta.loc[adata.obs_names, :]
adata.obsm['X_umap'] = meta.loc[adata.obs_names, ['wnnUMAP_1', 'wnnUMAP_2']].values
adata.obs = adata.obs.drop(columns=['wnnUMAP_1', 'wnnUMAP_2'])
for col in ['disease', 'celltype']:
    adata.obs[col] = adata.obs[col].astype('category')
print(adata)

In [ ]:
# print cell count per cell type
print(adata.obs['celltype'].value_counts())

N = 200  # down sample to max 200 per cell type

# downsample per cell type
np.random.seed(42)
sampled_indices = []

for ct in adata.obs['celltype'].unique():
    ct_indices = adata.obs_names[adata.obs['celltype'] == ct]
    n_sample = min(N, len(ct_indices))
    sampled = np.random.choice(ct_indices, size=n_sample, replace=False)
    sampled_indices.extend(sampled)

adata_sub = adata[sampled_indices].copy()

# Verify subset data
print(adata_sub.obs['celltype'].value_counts())

In [ ]:
# export to table for CIBERSORTx
print(f"Total genes: {adata_sub.n_vars}")
sc.pp.filter_genes(adata_sub, min_cells=10)  # expressed in at least 10 cells
sc.pp.filter_genes(adata_sub, min_counts=10)  # at least 10 total counts
adata_sub = adata_sub[:, ~adata_sub.var_names.str.startswith('mt-')].copy()
print(f"Genes remaining: {adata_sub.n_vars}")

X = adata_sub.X 

# Convert to dense array if sparse
X = X.toarray() if sp.issparse(X) else X

# Build the dataframe (genes x cells)
sc_ref = pd.DataFrame(
    X.T,  # transpose to genes x cells
    index=adata_sub.var_names,       # genes
    columns=adata_sub.obs['celltype'],  # cell type labels per cell
)

# Set index name to GeneSymbol as required by CIBERSORTx
sc_ref.index.name = 'GeneSymbol'

In [ ]:
# filter bulk-seq data for overlapping genes
bulk_count = pd.read_csv('tables_depo/count.csv', index_col=0)
bulk_count = bulk_count.T
# reorder the columns
cols = bulk_count.columns.tolist()
new_cols = cols[-4:] + cols[:-4]
bulk_count = bulk_count[new_cols]
bulk_count.index.name = 'Gene'

# find overlap
bulk_gene_list = set(bulk_count.index)
ref_gene_list = set(sc_ref.index)
overlap = ref_gene_list.intersection(bulk_gene_list)

print(f"Genes in bulk:            {len(bulk_gene_list)}")
print(f"Genes in sc reference:    {len(ref_gene_list)}")
print(f"Overlapping genes:        {len(overlap)}")
print(f"Bulk-only genes:          {len(bulk_gene_list - ref_gene_list)}")
print(f"Reference-only genes:     {len(ref_gene_list - bulk_gene_list)}")

# Filter both to overlapping genes
bulk_count = bulk_count.loc[list(overlap)]
sc_ref = sc_ref.loc[list(overlap)]

In [ ]:
# Save as tab-delimited file for CIBERSORTx
sc_ref.to_csv('cibersortx_reference_Humphreys_mus.txt', sep='\t')
bulk_count.to_csv('cibersortx_bulk_DFMO.txt', sep='\t')